# LAMBADA test-only activation cache

Generate paired test activations using the same dataset revision, model/tokenizer revisions,
text-boundary rules, context cap, precision, and attention implementation as the completed
training/validation extraction. The existing training and validation artifacts are read only.
Exported output heads are verified and referenced in place, without being copied or regenerated.

Run this notebook on a Colab GPU with the same shared Drive folder and Hugging Face access used
for the full extraction. Test artifacts are written beneath a separate, fingerprinted
`test_runs` folder inside the selected parent run. Completed shards can be resumed.
This notebook produces representations and labels; it does not train a stitch or compute test scores.


In [1]:
%pip install -q \
    "transformers>=5,<6" \
    "huggingface_hub>=1.32,<2" \
    "datasets>=3,<5" \
    "accelerate>=1.0" \
    safetensors \
    tqdm \
    "nltk>=3.9.1,<4" \
    numpy \
    matplotlib


In [2]:
import gc, hashlib, json, os, re
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import AutoModel, AutoTokenizer
from tqdm.auto import tqdm
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## Select the completed parent run

These defaults identify the completed 10,000-training / 4,869-validation run. `TEST_LIMIT=None`
selects every accepted test passage. The output directory is derived automatically; the parent
run's files are never rewritten. Lower `BATCH_SIZE` if needed without discarding completed shards.


In [3]:
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/coms-4705-project")
TRAIN_DIR = DRIVE_PROJECT_DIR / "datasets/lambada/train/7b6847a7f51353d0"
PARENT_CACHE_RUN_DIR = (DRIVE_PROJECT_DIR / "activations/lambada/7b6847a7f51353d0"
                        / "train10000_validation4869_a938da615918")
TEST_LIMIT = None
BATCH_SIZE = 8
SHARD_SIZE = 256
HIDDEN_SIZES = {"llama": 3072, "qwen": 2048}

if TEST_LIMIT is not None and (type(TEST_LIMIT) is not int or TEST_LIMIT <= 0):
    raise ValueError("TEST_LIMIT must be None or a positive integer")
if type(BATCH_SIZE) is not int or BATCH_SIZE <= 0 or type(SHARD_SIZE) is not int or SHARD_SIZE <= 0:
    raise ValueError("Batch and shard sizes must be positive integers")


## Artifact verification and checkpoint helpers


In [4]:
def save_tensor_file(payload, path):
    from tempfile import NamedTemporaryFile
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(dir=path.parent, delete=False) as tmp:
        temp_path = Path(tmp.name)
    try:
        torch.save(payload, temp_path)
        os.replace(temp_path, path)
    finally:
        temp_path.unlink(missing_ok=True)

def save_json(value, path):
    from tempfile import NamedTemporaryFile
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name)
        json.dump(value, handle, indent=2)
        handle.write("\n")
    try:
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

def file_sha256(path):
    result = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            result.update(block)
    return result.hexdigest()

def object_sha256(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

def save_checked_tensor(payload, path):
    path = Path(path)
    save_tensor_file(payload, path)
    save_json({"sha256": file_sha256(path)}, path.with_suffix(".sha256.json"))

def load_checked_tensor(path):
    path = Path(path)
    checksum = path.with_suffix(".sha256.json")
    if not path.is_file() or not checksum.is_file():
        return None
    if json.loads(checksum.read_text())["sha256"] != file_sha256(path):
        return None
    return torch.load(path, map_location="cpu", weights_only=True)

def checked_vectors(path, ids, fingerprint, width):
    payload = load_checked_tensor(path)
    if payload is None:
        return None
    if payload.get("ids") != ids or payload.get("fingerprint") != fingerprint:
        raise ValueError(f"Incompatible IDs/provenance: {path}")
    for stage in ("pre", "post"):
        matrix = payload.get(stage)
        if not isinstance(matrix, torch.Tensor) or matrix.shape != (len(ids), width):
            raise ValueError(f"Invalid tensor shape: {path}/{stage}")
        if not matrix.is_floating_point() or not torch.isfinite(matrix).all():
            raise ValueError(f"Invalid dtype/nonfinite values: {path}/{stage}")
    return payload

def extract_checkpointed(name, backbone, tokenizer, rows, directory, fingerprint, device):
    ids = [row["id"] for row in rows]
    final_path = Path(directory) / f"{name}.pt"
    if checked_vectors(final_path, ids, fingerprint, HIDDEN_SIZES[name]) is not None:
        return final_path
    shards = []
    for start in tqdm(range(0, len(rows), SHARD_SIZE), desc=f"{name}: {Path(directory).name}"):
        batch_rows = rows[start:start + SHARD_SIZE]
        shard_ids = [row["id"] for row in batch_rows]
        path = Path(directory) / "shards" / name / f"{start:08d}.pt"
        payload = checked_vectors(path, shard_ids, fingerprint, HIDDEN_SIZES[name])
        if payload is None:
            if backbone is None:
                raise RuntimeError("A model is required to extract incomplete shards")
            tensors = extract(name, backbone, tokenizer, batch_rows, device)
            if not all(torch.isfinite(v).all() for v in tensors.values()):
                raise ValueError("Nonfinite extracted activations")
            payload = dict(ids=shard_ids, fingerprint=fingerprint, batch_size=BATCH_SIZE, **tensors)
            save_checked_tensor(payload, path)
        shards.append(path)
    payloads = [load_checked_tensor(path) for path in shards]
    # Keeping only final-position vectors bounds CPU storage independently of context lengths.
    combined = dict(ids=ids, fingerprint=fingerprint,
                    pre=torch.cat([p["pre"] for p in payloads]),
                    post=torch.cat([p["post"] for p in payloads]),
                    shard_batch_sizes=[p["batch_size"] for p in payloads])
    save_checked_tensor(combined, final_path)
    return final_path


In [5]:
def require_tensor_artifact(path):
    payload = load_checked_tensor(path)
    if payload is None:
        raise ValueError(f"Missing or checksum-invalid parent artifact: {path}")
    return payload


def verify_parent_run(training_dir, parent_dir):
    training_dir, parent_dir = Path(training_dir), Path(parent_dir)
    training_meta = json.loads((training_dir / "provenance.json").read_text())
    if file_sha256(training_dir / "examples.jsonl") != training_meta["manifest_sha256"]:
        raise ValueError("Training manifest checksum mismatch")
    run = json.loads((parent_dir / "run.json").read_text())
    completed = json.loads((parent_dir / "completed.json").read_text())
    identity = run["identity"]
    if completed.get("identity") != identity or not {"train", "validation"}.issubset(identity["split_fingerprints"]):
        raise ValueError("Select a completed training/validation extraction run")
    if identity["models"] != training_meta["models"] or identity["revisions"] != training_meta["revisions"]:
        raise ValueError("Parent run and training metadata use different model/tokenizer revisions")
    if set(identity["models"]) != set(HIDDEN_SIZES):
        raise ValueError("Expected Llama and Qwen model identities")
    if run["settings"]["padding_side"] != "right" or run["settings"]["extraction_point"] != "final_token_pre_and_post_rmsnorm":
        raise ValueError("Parent extraction settings do not match this test-cache protocol")
    parent_manifest = (parent_dir / "train/examples.jsonl").read_bytes()
    parent_train_count = len(parent_manifest.splitlines())
    expected_train_info = dict(training_meta, split="train", n_examples=parent_train_count,
        manifest_sha256=hashlib.sha256(parent_manifest).hexdigest(),
        extraction=run["settings"], environment=run["environment"])
    if object_sha256(expected_train_info) != identity["split_fingerprints"]["train"]:
        raise ValueError("Parent training manifest does not match the selected dataset/provenance")
    parent_hash = object_sha256(identity)
    head_refs = {}
    for name, repo in identity["models"].items():
        path = parent_dir / f"{name}_frozen_head.pt"
        state = require_tensor_artifact(path)
        expected_head_identity = dict(model=repo, revision=identity["revisions"][name], run_hash=parent_hash,
            compute_dtype=run["settings"]["compute_dtype"], attention=run["settings"]["attention"])
        report = json.loads((parent_dir / f"{name}_head_validation.json").read_text())
        observations = report.get("observations", [])
        if state.get("provenance") != expected_head_identity or report.get("provenance") != expected_head_identity:
            raise ValueError(f"Wrong parent head provenance: {name}")
        if report.get("passed") is not True or not observations or not all(o.get("allclose") and o.get("top1_match") for o in observations):
            raise ValueError(f"Parent {name} head reconstruction checks did not pass")
        if state["hidden_size"] != HIDDEN_SIZES[name]:
            raise ValueError(f"Parent {name} hidden size mismatch")
        head_refs[name] = dict(path=str(path), sha256=file_sha256(path),
            provenance=expected_head_identity, vocab_size=state["vocab_size"],
            validation_report=str(parent_dir / f"{name}_head_validation.json"))
        del state
        gc.collect()
    return training_meta, run, parent_hash, head_refs


meta, parent_run, parent_run_hash, head_references = verify_parent_run(TRAIN_DIR, PARENT_CACHE_RUN_DIR)
MODELS, revisions = meta["models"], meta["revisions"]
MAX_TOKENS = meta["max_tokens"]
dtype_map = {"torch.bfloat16": torch.bfloat16, "torch.float16": torch.float16, "torch.float32": torch.float32}
if parent_run["settings"]["compute_dtype"] not in dtype_map:
    raise ValueError("Unsupported parent compute dtype")
compute_dtype = dtype_map[parent_run["settings"]["compute_dtype"]]
ATTENTION_IMPLEMENTATION = parent_run["settings"]["attention"]
print("Verified parent run:", PARENT_CACHE_RUN_DIR)
print("Model/tokenizer revisions:", revisions)
print("Reusing exported head files:", {name: item["path"] for name, item in head_references.items()})


Verified parent run: /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918
Model/tokenizer revisions: {'llama': '13afe5124825b4f3751f836b40dafda64c1ed062', 'qwen': '3aab1f1954e9cc14eb9509a215f9e5ca08227a9b'}
Reusing exported head files: {'llama': '/content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/llama_frozen_head.pt', 'qwen': '/content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/qwen_frozen_head.pt'}


## Authenticate and load the pinned tokenizers


In [6]:
from huggingface_hub import login

token = os.environ.get("HF_TOKEN")
if not token:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
login(token=token)
del token
tokenizers = {name: AutoTokenizer.from_pretrained(repo, revision=revisions[name])
              for name, repo in MODELS.items()}
for tok in tokenizers.values():
    tok.padding_side = "right"
    if tok.eos_token_id is None:
        raise ValueError("The extraction protocol requires an EOS padding token")
    tok.pad_token = tok.eos_token


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## Build the test manifest

Load only the official test split. Preserve passage whitespace and derive both models' labels
from checked full-text token boundaries, using the same filtering rules as validation. Retain
repeated contexts in the official split and give each source row a unique `test/<index>` ID.
Record rejection reasons and sample counts alongside the manifest.


In [7]:
WORD = re.compile(r"\S+")
LETTER_WORD = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def prepare_test_rows(source_rows, tokenizers, max_tokens, limit=None):
    accepted, rejected = [], Counter()
    examined = 0
    for index, source_row in enumerate(source_rows):
        examined += 1
        text = source_row["text"]
        spans = list(WORD.finditer(text))
        if len(spans) < 2 or spans[-1].end() != len(text) or LETTER_WORD.fullmatch(spans[-1].group()) is None:
            rejected["passage_format"] += 1
            continue
        boundary = spans[-2].end()
        row = dict(id=f"test/{index}", split="test", n_words=len(spans),
                   prefix=text[:boundary], continuation=text[boundary:])
        reason = None
        for name, tok in tokenizers.items():
            prefix_ids = tok.encode(row["prefix"])
            full_ids = tok.encode(text)
            if not prefix_ids or len(prefix_ids) > max_tokens:
                reason = f"{name}:empty_or_overlength_prefix"
                break
            if full_ids[:len(prefix_ids)] != prefix_ids or len(full_ids) <= len(prefix_ids):
                reason = f"{name}:unstable_or_empty_continuation"
                break
            row.update({name+"_length": len(prefix_ids), name+"_target_id": full_ids[len(prefix_ids)],
                        name+"_target_length": len(full_ids)-len(prefix_ids)})
        if reason is not None:
            rejected[reason] += 1
            continue
        accepted.append(row)
        if limit is not None and len(accepted) >= limit:
            break
    if not accepted or (limit is not None and len(accepted) < limit):
        raise ValueError(f"Only {len(accepted)} accepted test examples; requested {limit}")
    return accepted, dict(examined=examined, accepted=len(accepted), rejected=dict(rejected),
                          total_source_rows=len(source_rows), requested=limit)


test_dataset = load_dataset(meta["dataset"], revision=meta["dataset_revision"], split="test")
test_rows, test_accounting = prepare_test_rows(test_dataset, tokenizers, MAX_TOKENS, TEST_LIMIT)
for name in MODELS:
    if any(not 0 <= row[f"{name}_target_id"] < head_references[name]["vocab_size"] for row in test_rows):
        raise ValueError(f"Test labels outside the exported {name} vocabulary")
print("Test preprocessing:", test_accounting)


README.md:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

plain_text/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  269MB            

plain_text/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

plain_text/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  281MB            

plain_text/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.14MB            

... (3 outputs left out)

Generating train split:   0%|          | 0/2662 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5153 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4869 [00:00<?, ? examples/s]

Test preprocessing: {'examined': 5153, 'accepted': 5153, 'rejected': {}, 'total_source_rows': 5153, 'requested': None}


## Test-only extraction

Use the parent's precision and attention implementation, then run one backbone at a time.
No training/validation inference or output-head export is performed. Batch size may be lowered
on resume; shard size and the recorded artifact identity must remain fixed. No test metrics are computed.


In [8]:
@torch.inference_mode()
def extract(name, model, tokenizer, rows, device):
    collected = {"pre": [], "post": []}
    for start in range(0, len(rows), BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        inputs = tokenizer([row["prefix"] for row in batch], padding=True,
                           return_tensors="pt").to(device)
        lengths = inputs.attention_mask.sum(1)
        assert lengths.tolist() == [row[name + "_length"] for row in batch]
        idx = (torch.arange(len(batch), device=device), lengths - 1)

        def capture(module, args, output):
            collected["pre"].append(args[0][idx].cpu())
            collected["post"].append(output[idx].cpu())

        hook = model.norm.register_forward_hook(capture)
        try:
            out = model(**inputs, use_cache=False)
        finally:
            hook.remove()
        if start == 0:
            post = collected["post"][-1]
            torch.testing.assert_close(post, out.last_hidden_state[idx].cpu())
            torch.testing.assert_close(model.norm(collected["pre"][-1].to(device)).cpu(), post)
        del out
    return {stage: torch.cat(chunks) for stage, chunks in collected.items()}


In [9]:
def assemble_cache(split, rows, run_dir, fingerprint, provenance):
    ids = [row["id"] for row in rows]
    cache = dict(
        ids=ids,
        target_token_ids=torch.tensor([row["qwen_target_id"] for row in rows]),
        llama_target_token_ids=torch.tensor([row["llama_target_id"] for row in rows]),
        source_model=MODELS["llama"],
        target_model=MODELS["qwen"],
        provenance=provenance,
    )
    for name, width in HIDDEN_SIZES.items():
        part = torch.load(run_dir / f"{name}.pt", map_location="cpu", weights_only=True)
        assert part["ids"] == ids and part["fingerprint"] == fingerprint
        for stage, suffix in [("pre", ""), ("post", "_post")]:
            matrix = part[stage]
            assert matrix.shape == (len(rows), width) and torch.isfinite(matrix).all()
            cache["h_" + name + suffix] = matrix
    cache_path = run_dir / "paired.pt"
    save_checked_tensor(cache, cache_path)
    saved = torch.load(cache_path, map_location="cpu", weights_only=True)
    assert saved["ids"] == ids
    for key, value in cache.items():
        if isinstance(value, torch.Tensor):
            assert torch.equal(saved[key], value)
            print(key, tuple(value.shape), value.dtype)
    print("Paired cache:", cache_path)
    return cache_path


In [10]:
def test_cache_provenance(training_meta, parent_hash, parent_settings, parent_heads, rows, accounting, environment, shard_size):
    manifest = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows).encode()
    extraction = dict(parent_settings, schema="paired-test-only-v1", shard_size=shard_size)
    info = dict(dataset=training_meta["dataset"], dataset_revision=training_meta["dataset_revision"],
                split="test", models=training_meta["models"], revisions=training_meta["revisions"],
                max_tokens=training_meta["max_tokens"], n_examples=len(rows),
                manifest_sha256=hashlib.sha256(manifest).hexdigest(), parent_run_hash=parent_hash,
                extraction=extraction, head_references=parent_heads, accounting=accounting, environment=environment)
    return manifest, info


def extract_test_models(rows, directory, fingerprint, meta, tokenizers, dtype, attention, device):
    ids = [row["id"] for row in rows]
    for name, repo in meta["models"].items():
        path = Path(directory) / f"{name}.pt"
        if checked_vectors(path, ids, fingerprint, HIDDEN_SIZES[name]) is not None:
            print(f"{name}: verified test cache reused")
            continue
        all_shards_complete = all(
            checked_vectors(Path(directory) / "shards" / name / f"{start:08d}.pt",
                            ids[start:start+SHARD_SIZE], fingerprint, HIDDEN_SIZES[name]) is not None
            for start in range(0, len(rows), SHARD_SIZE)
        )
        model = None
        try:
            if not all_shards_complete:
                model = AutoModel.from_pretrained(repo, revision=meta["revisions"][name], dtype=dtype,
                                                 attn_implementation=attention)
                model = model.eval().requires_grad_(False).to(device)
                if model.config.hidden_size != HIDDEN_SIZES[name]:
                    raise ValueError(f"Unexpected {name} hidden size")
            extract_checkpointed(name, model, tokenizers[name], rows, directory, fingerprint, device)
        finally:
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


def paired_test_matches(path, rows, provenance, fingerprint):
    paired = load_checked_tensor(path)
    if paired is None:
        return False
    ids = [row["id"] for row in rows]
    if paired.get("provenance") != provenance or paired.get("ids") != ids:
        raise ValueError("Existing paired test cache has incompatible provenance or IDs")
    if paired.get("source_model") != MODELS["llama"] or paired.get("target_model") != MODELS["qwen"]:
        raise ValueError("Existing paired test cache has different model identities")
    for name in ("llama", "qwen"):
        part = checked_vectors(Path(path).parent / f"{name}.pt", ids, fingerprint, HIDDEN_SIZES[name])
        if part is None:
            return False
        label_key = "target_token_ids" if name == "qwen" else "llama_target_token_ids"
        expected = torch.tensor([row[f"{name}_target_id"] for row in rows], dtype=torch.int64)
        if paired[label_key].dtype != torch.int64 or not torch.equal(paired[label_key], expected):
            return False
        for stage, suffix in (("pre", ""), ("post", "_post")):
            matrix = paired[f"h_{name}{suffix}"]
            if str(matrix.dtype) != provenance["extraction"]["compute_dtype"] or not torch.equal(matrix, part[stage]):
                return False
    return True


def run_test_cache(rows, training_meta, parent_dir, parent_hash, parent_settings, parent_heads, accounting,
                   tokenizers, dtype, attention, environment, device="cuda"):
    manifest, provenance = test_cache_provenance(training_meta, parent_hash, parent_settings, parent_heads,
                                               rows, accounting, environment, SHARD_SIZE)
    fingerprint = object_sha256(provenance)
    test_root = Path(parent_dir) / "test_runs" / f"test{len(rows)}_{fingerprint[:12]}"
    test_dir = test_root / "test"
    test_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = test_dir / "examples.jsonl"
    if manifest_path.exists() and manifest_path.read_bytes() != manifest:
        raise ValueError("Existing output directory contains a different test manifest")
    manifest_path.write_bytes(manifest)
    identity = dict(split_fingerprints={"test":fingerprint}, models=training_meta["models"], revisions=training_meta["revisions"])
    save_json(dict(identity=identity, settings=provenance["extraction"], environment=environment,
                   parent_run_hash=parent_hash, parent_artifact_dir=str(parent_dir), head_references=parent_heads), test_root/"run.json")
    print(f"test: {len(rows):,} examples -> {test_dir}")
    extract_test_models(rows, test_dir, fingerprint, training_meta, tokenizers, dtype, attention, device)
    cache_path = test_dir / "paired.pt"
    if paired_test_matches(cache_path, rows, provenance, fingerprint):
        print("Verified paired test cache reused:", cache_path)
    else:
        cache_path = assemble_cache("test", rows, test_dir, fingerprint, provenance)
    save_json(dict(identity=identity, parent_run_hash=parent_hash, paired_paths={"test":str(cache_path)}), test_root/"completed.json")
    return test_root, cache_path


if not torch.cuda.is_available():
    raise RuntimeError("Choose a CUDA GPU runtime before test extraction")
if compute_dtype == torch.bfloat16 and not torch.cuda.is_bf16_supported():
    raise RuntimeError("Choose a GPU supporting bfloat16 to match the completed parent extraction")
environment = {name: version(name) for name in ("torch", "transformers", "huggingface_hub", "datasets", "accelerate")}
TEST_CACHE_RUN_DIR, TEST_CACHE_PATH = run_test_cache(
    test_rows, meta, PARENT_CACHE_RUN_DIR, parent_run_hash, parent_run["settings"], head_references,
    test_accounting, tokenizers, compute_dtype, ATTENTION_IMPLEMENTATION, environment,
)
print("\nUse these artifact locations for later final test evaluation:")
print(f"PARENT_CACHE_RUN_DIR = Path({str(PARENT_CACHE_RUN_DIR)!r})")
print(f"TEST_CACHE_RUN_DIR = Path({str(TEST_CACHE_RUN_DIR)!r})")
print(f"TEST_CACHE_PATH = Path({str(TEST_CACHE_PATH)!r})")


test: 5,153 examples -> /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/test_runs/test5153_d75faa265fa6/test


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

llama: test:   0%|          | 0/21 [00:00<?, ?it/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

qwen: test:   0%|          | 0/21 [00:00<?, ?it/s]

target_token_ids (5153,) torch.int64
llama_target_token_ids (5153,) torch.int64
h_llama (5153, 3072) torch.bfloat16
h_llama_post (5153, 3072) torch.bfloat16
h_qwen (5153, 2048) torch.bfloat16
h_qwen_post (5153, 2048) torch.bfloat16
Paired cache: /content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/test_runs/test5153_d75faa265fa6/test/paired.pt

Use these artifact locations for later final test evaluation:
PARENT_CACHE_RUN_DIR = Path('/content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918')
TEST_CACHE_RUN_DIR = Path('/content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/test_runs/test5153_d75faa265fa6')
TEST_CACHE_PATH = Path('/content/drive/MyDrive/coms-4705-project/activations/lambada/7b6847a7f51353d0/train10000_validation4869_a938da615918/test_runs/test5153_d75faa265fa6/test/paired.pt')
